# BioMini Phase I Educational Materials

## Notebook 03 — DNA, RNA, Protein and Biological Transformations
### Sequence Object에서 Biological Transformation으로

Notebook 02에서는 DNA, RNA, Protein에 공통적인 sequence handling을
`SequenceMolecule` abstraction으로 정리했습니다.

이번 Notebook에서는 그 객체들이 단순히 sequence를 저장하는 데 그치지 않고,
**생물학적 transformation을 수행하는 domain object**로 발전합니다.

핵심 흐름은 다음과 같습니다.

```text
DNA
 ↓ transcription
RNA
 ↓ translation
Protein
```

그리고 DNA에서는:

```text
DNA
 ├── complement
 └── reverse complement
```

를 구현합니다.

이 Notebook은 BioMini Phase I의 **Step 4 — Biological Transformations**를
교육용으로 재구성합니다.

---

## Learning Objectives

이 Notebook을 마치면 다음을 할 수 있어야 합니다.

1. DNA complement와 reverse complement를 구현한다.
2. transcription을 문자열 치환이 아니라 domain-object transformation으로 이해한다.
3. RNA codon을 이용해 translation을 구현한다.
4. stop codon 처리 정책을 설명한다.
5. incomplete terminal codon 처리 정책을 설명한다.
6. biological assumption을 software contract로 명시할 수 있다.
7. transformation 결과를 새로운 biological object로 반환하는 이유를 설명한다.
8. empty translation product가 왜 중요한 edge case인지 이해한다.
9. canonical BioMini의 transformation semantics와 교육용 구현을 비교한다.

# 1. Starting Point — Notebook 02의 Hierarchy

먼저 이번 Notebook에서 사용할 최소 hierarchy를 다시 준비합니다.

교육 목적상 Notebook 03은 독립적으로 실행할 수 있도록 필요한 class를 포함합니다.
다만 실제 프로젝트에서는 production source를 중복해서 관리하지 않습니다.

In [ ]:
class ValidationError(ValueError):
    pass


class Molecule:
    def __init__(self, name: str):
        if not isinstance(name, str) or not name.strip():
            raise ValidationError("name must be a non-empty string")
        self.name = name.strip()

    def get_name(self) -> str:
        return self.name


class SequenceMolecule(Molecule):
    allow_empty_sequence = False

    def __init__(self, name: str, sequence: str):
        super().__init__(name)

        if not isinstance(sequence, str):
            raise ValidationError("sequence must be a string")

        normalized = "".join(sequence.split()).upper()

        if not normalized and not self.allow_empty_sequence:
            raise ValidationError("sequence must be a non-empty string")

        self.sequence = normalized

    def get_sequence(self) -> str:
        return self.sequence

    def __len__(self) -> int:
        return len(self.sequence)

    def __repr__(self) -> str:
        return (
            f"{self.__class__.__name__}("
            f"name='{self.name}', sequence='{self.sequence}')"
        )


class NucleicAcid(SequenceMolecule):
    pass

Protein은 translation 결과가 될 수 있으므로,
이번 Notebook에서는 canonical BioMini와 같은 policy를 반영해
**empty Protein을 허용**합니다.

이 결정은 뒤에서 다시 설명합니다.

In [ ]:
class AminoAcid:
    acceptable_codes = set("ARNDCQEGHILKMFPSTWYV")


class Protein(SequenceMolecule):
    allow_empty_sequence = True
    valid_codes = AminoAcid.acceptable_codes

    def __init__(self, name: str, sequence: str):
        super().__init__(name, sequence)
        self._validate_sequence()

    def _validate_sequence(self) -> None:
        invalid_codes = set(self.sequence) - self.valid_codes

        if invalid_codes:
            raise ValidationError(
                f"invalid protein code(s): {sorted(invalid_codes)}"
            )

DNA와 RNA도 준비합니다.

In [ ]:
class DNA(NucleicAcid):
    valid_codes = set("ATGC")

    def __init__(self, name: str, sequence: str):
        super().__init__(name, sequence)
        self._validate_sequence()

    def _validate_sequence(self) -> None:
        invalid_codes = set(self.sequence) - self.valid_codes

        if invalid_codes:
            raise ValidationError(
                f"invalid DNA code(s): {sorted(invalid_codes)}"
            )


class RNA(NucleicAcid):
    valid_codes = set("AUGC")

    def __init__(self, name: str, sequence: str):
        super().__init__(name, sequence)
        self._validate_sequence()

    def _validate_sequence(self) -> None:
        invalid_codes = set(self.sequence) - self.valid_codes

        if invalid_codes:
            raise ValidationError(
                f"invalid RNA code(s): {sorted(invalid_codes)}"
            )

In [ ]:
dna = DNA("example_dna", "ATGGCCTTT")
rna = RNA("example_rna", "AUGGCCUUU")
protein = Protein("example_protein", "MAF")

print(dna)
print(rna)
print(protein)

# 2. DNA Complement

DNA에서는 base pairing rule을 사용할 수 있습니다.

```text
A ↔ T
G ↔ C
```

예:

```text
5' - ATGC - 3'
     ||||
3' - TACG - 5'
```

software에서는 character mapping으로 구현할 수 있습니다.

In [ ]:
def dna_complement(sequence: str) -> str:
    table = str.maketrans(
        "ATGC",
        "TACG"
    )
    return sequence.translate(table)

In [ ]:
sequence = "ATGC"

print("original   :", sequence)
print("complement :", dna_complement(sequence))

함수로도 구현할 수 있지만, 이미 `DNA`라는 domain object가 있습니다.

따라서 complement는 DNA와 밀접한 behavior이므로
`DNA` method로 이동하는 것이 자연스럽습니다.

In [ ]:
class DNA(NucleicAcid):
    valid_codes = set("ATGC")

    def __init__(self, name: str, sequence: str):
        super().__init__(name, sequence)
        self._validate_sequence()

    def _validate_sequence(self) -> None:
        invalid_codes = set(self.sequence) - self.valid_codes

        if invalid_codes:
            raise ValidationError(
                f"invalid DNA code(s): {sorted(invalid_codes)}"
            )

    def complement(self) -> str:
        table = str.maketrans(
            "ATGC",
            "TACG"
        )
        return self.sequence.translate(table)

In [ ]:
dna = DNA(
    "dna1",
    "ATGC"
)

print("DNA        :", dna.get_sequence())
print("complement :", dna.complement())

## Design Point

여기서 `complement()`가 문자열을 반환하는 점에 주목하십시오.

현재 Phase I canonical BioMini도 complement와 reverse complement는
문자열을 반환합니다.

즉 모든 method가 반드시 새로운 domain object를 반환해야 하는 것은 아닙니다.

어떤 operation이 새로운 biological entity를 만들어내는지,
어떤 operation이 단순 derived representation인지 구분해야 합니다.

# 3. Reverse Complement

DNA sequence를 다룰 때는 reverse complement가 자주 필요합니다.

구현은 다음 두 단계를 결합합니다.

```text
DNA
 ↓ complement
complement sequence
 ↓ reverse
reverse complement
```

In [ ]:
class DNA(NucleicAcid):
    valid_codes = set("ATGC")

    def __init__(self, name: str, sequence: str):
        super().__init__(name, sequence)
        self._validate_sequence()

    def _validate_sequence(self) -> None:
        invalid_codes = set(self.sequence) - self.valid_codes

        if invalid_codes:
            raise ValidationError(
                f"invalid DNA code(s): {sorted(invalid_codes)}"
            )

    def complement(self) -> str:
        table = str.maketrans(
            "ATGC",
            "TACG"
        )
        return self.sequence.translate(table)

    def reverse_complement(self) -> str:
        return self.complement()[::-1]

In [ ]:
dna = DNA(
    "dna2",
    "ATGCCG"
)

print("original           :", dna.get_sequence())
print("complement         :", dna.complement())
print("reverse complement :", dna.reverse_complement())

# 4. From DNA to RNA — Transcription

이제 단순 sequence operation에서 **object transformation**으로 넘어갑니다.

Phase I에서는 저장된 DNA를 **coding strand**로 간주합니다.

따라서 transcription은 간단히:

```text
T → U
```

로 표현합니다.

예:

```text
DNA: ATGGCCTTT
RNA: AUGGCCUUU
```

In [ ]:
def transcribe_sequence(dna_sequence: str) -> str:
    return dna_sequence.replace(
        "T",
        "U"
    )

In [ ]:
print(
    transcribe_sequence(
        "ATGGCCTTT"
    )
)

하지만 BioMini의 관점에서 중요한 것은 단순히 문자열 `"AUGGCCUUU"`를 얻는 것이 아닙니다.

DNA에서 RNA가 생성되었으므로 결과는 새로운 `RNA` object가 되는 것이 자연스럽습니다.

```text
DNA object
   ↓ transcribe()
RNA object
```

먼저 `RNA` class를 준비합니다.

In [ ]:
class RNA(NucleicAcid):
    valid_codes = set("AUGC")

    def __init__(self, name: str, sequence: str):
        super().__init__(name, sequence)
        self._validate_sequence()

    def _validate_sequence(self) -> None:
        invalid_codes = set(self.sequence) - self.valid_codes

        if invalid_codes:
            raise ValidationError(
                f"invalid RNA code(s): {sorted(invalid_codes)}"
            )

이제 DNA에 `transcribe()`를 추가합니다.

In [ ]:
class DNA(NucleicAcid):
    valid_codes = set("ATGC")

    def __init__(self, name: str, sequence: str):
        super().__init__(name, sequence)
        self._validate_sequence()

    def _validate_sequence(self) -> None:
        invalid_codes = set(self.sequence) - self.valid_codes

        if invalid_codes:
            raise ValidationError(
                f"invalid DNA code(s): {sorted(invalid_codes)}"
            )

    def complement(self) -> str:
        table = str.maketrans(
            "ATGC",
            "TACG"
        )
        return self.sequence.translate(table)

    def reverse_complement(self) -> str:
        return self.complement()[::-1]

    def transcribe(self):
        return RNA(
            self.name + "_RNA",
            self.sequence.replace("T", "U")
        )

In [ ]:
dna = DNA(
    "gene1",
    "ATGGCCTTT"
)

rna = dna.transcribe()

print("DNA:", dna)
print("RNA:", rna)

print(
    type(rna).__name__
)

여기서 중요한 설계 포인트는 다음입니다.

```text
input
DNA object

output
RNA object
```

즉 transcription은 단순한 문자열 치환이 아니라
**domain object에서 domain object로의 transformation**입니다.

# 5. RNA Translation

이제 RNA sequence를 Protein sequence로 변환합니다.

RNA는 3개의 nucleotide씩 codon으로 읽습니다.

예:

```text
AUG GCC UUU
 ↓   ↓   ↓
 M   A   F
```

따라서 먼저 codon table이 필요합니다.

In [ ]:
CODON_TABLE = {
    "UUU":"F","UUC":"F","UUA":"L","UUG":"L",
    "UCU":"S","UCC":"S","UCA":"S","UCG":"S",
    "UAU":"Y","UAC":"Y","UAA":"*","UAG":"*",
    "UGU":"C","UGC":"C","UGA":"*","UGG":"W",

    "CUU":"L","CUC":"L","CUA":"L","CUG":"L",
    "CCU":"P","CCC":"P","CCA":"P","CCG":"P",
    "CAU":"H","CAC":"H","CAA":"Q","CAG":"Q",
    "CGU":"R","CGC":"R","CGA":"R","CGG":"R",

    "AUU":"I","AUC":"I","AUA":"I","AUG":"M",
    "ACU":"T","ACC":"T","ACA":"T","ACG":"T",
    "AAU":"N","AAC":"N","AAA":"K","AAG":"K",
    "AGU":"S","AGC":"S","AGA":"R","AGG":"R",

    "GUU":"V","GUC":"V","GUA":"V","GUG":"V",
    "GCU":"A","GCC":"A","GCA":"A","GCG":"A",
    "GAU":"D","GAC":"D","GAA":"E","GAG":"E",
    "GGU":"G","GGC":"G","GGA":"G","GGG":"G",
}

## Codon 단위 iteration

3개씩 이동합니다.

```python
range(0, len(sequence) - 2, 3)
```

`-2`를 사용하는 이유는 길이가 3보다 짧은 마지막 fragment를 자동으로 제외하기 위해서입니다.

In [ ]:
rna_sequence = "AUGGCCUUU"

for i in range(
    0,
    len(rna_sequence) - 2,
    3
):
    codon = rna_sequence[i:i + 3]
    print(
        codon,
        "->",
        CODON_TABLE[codon]
    )

# 6. Stop Codon Policy

Phase I에서는 다음 세 codon을 stop으로 처리합니다.

```text
UAA
UAG
UGA
```

codon table에서는 `*`로 표현합니다.

정책은:

> 첫 stop codon을 만나면 translation을 종료한다.

예:

```text
AUG GCC UAA GGG
 ↓   ↓   STOP
 M   A
```

결과 Protein sequence는 `"MA"`입니다.

In [ ]:
def translate_sequence(rna_sequence: str) -> str:
    protein = []

    for i in range(
        0,
        len(rna_sequence) - 2,
        3
    ):
        codon = rna_sequence[i:i + 3]
        amino_acid = CODON_TABLE[codon]

        if amino_acid == "*":
            break

        protein.append(amino_acid)

    return "".join(protein)

In [ ]:
print(
    translate_sequence(
        "AUGGCCUAAGGG"
    )
)

# 7. Incomplete Terminal Codon Policy

다음 RNA를 생각해 봅니다.

```text
AUG GCC UU
```

마지막 `UU`는 codon이 아닙니다.

Phase I policy는:

> incomplete terminal codon은 무시한다.

따라서 결과는:

```text
AUG → M
GCC → A
UU  → ignored
```

즉 `"MA"`입니다.

In [ ]:
sequence = "AUGGCCUU"

print(
    translate_sequence(
        sequence
    )
)

이 정책은 하나의 **software contract**입니다.

다른 software는 incomplete codon에 warning을 내거나 error를 발생시킬 수도 있습니다.

중요한 것은:

> biological assumption과 edge-case policy를 코드 속에 숨기지 않고 명시하는 것

입니다.

# 8. RNA.translate() — Protein Object 반환

이제 sequence function을 domain behavior로 옮깁니다.

In [ ]:
class RNA(NucleicAcid):
    valid_codes = set("AUGC")
    codon_table = CODON_TABLE

    def __init__(self, name: str, sequence: str):
        super().__init__(name, sequence)
        self._validate_sequence()

    def _validate_sequence(self) -> None:
        invalid_codes = set(self.sequence) - self.valid_codes

        if invalid_codes:
            raise ValidationError(
                f"invalid RNA code(s): {sorted(invalid_codes)}"
            )

    def translate(self):
        protein_sequence = []

        for i in range(
            0,
            len(self.sequence) - 2,
            3
        ):
            codon = self.sequence[i:i + 3]
            amino_acid = self.codon_table[codon]

            if amino_acid == "*":
                break

            protein_sequence.append(
                amino_acid
            )

        return Protein(
            self.name + "_protein",
            "".join(protein_sequence)
        )

DNA의 `transcribe()`는 이 새로운 RNA class를 사용하도록 다시 정의합니다.

In [ ]:
class DNA(NucleicAcid):
    valid_codes = set("ATGC")

    def __init__(self, name: str, sequence: str):
        super().__init__(name, sequence)
        self._validate_sequence()

    def _validate_sequence(self) -> None:
        invalid_codes = set(self.sequence) - self.valid_codes

        if invalid_codes:
            raise ValidationError(
                f"invalid DNA code(s): {sorted(invalid_codes)}"
            )

    def complement(self) -> str:
        table = str.maketrans(
            "ATGC",
            "TACG"
        )
        return self.sequence.translate(table)

    def reverse_complement(self) -> str:
        return self.complement()[::-1]

    def transcribe(self):
        return RNA(
            self.name + "_RNA",
            self.sequence.replace(
                "T",
                "U"
            )
        )

# 9. Complete Biological Flow

이제 DNA → RNA → Protein 전체 flow를 실행할 수 있습니다.

In [ ]:
dna = DNA(
    "gene1",
    "ATGGCCTTT"
)

rna = dna.transcribe()
protein = rna.translate()

print("DNA     :", dna)
print("RNA     :", rna)
print("Protein :", protein)

print(
    "Protein sequence:",
    protein.get_sequence()
)

예상 biological flow:

```text
DNA
ATG GCC TTT
 ↓
RNA
AUG GCC UUU
 ↓
Protein
M A F
```

# 10. Stop Codon Example

In [ ]:
dna = DNA(
    "gene_with_stop",
    "ATGGCCTAAGGG"
)

rna = dna.transcribe()
protein = rna.translate()

print("RNA     :", rna.get_sequence())
print("Protein :", protein.get_sequence())

DNA:

```text
ATG GCC TAA GGG
```

RNA:

```text
AUG GCC UAA GGG
```

translation은 `UAA`에서 종료됩니다.

따라서 결과:

```text
MA
```

입니다.

# 11. Empty Translation Product

이제 중요한 edge case를 다룹니다.

RNA가 stop codon으로 시작한다고 가정합니다.

```text
RNA = UAA
```

첫 codon에서 translation이 즉시 종료되므로 amino acid는 하나도 생성되지 않습니다.

결과는:

```text
Protein sequence = ""
```

입니다.

이것은 software error가 아니라 **valid computational result**일 수 있습니다.

In [ ]:
rna = RNA(
    "stop_only",
    "UAA"
)

protein = rna.translate()

print(
    "Protein sequence:",
    repr(
        protein.get_sequence()
    )
)

print(
    "Protein length:",
    len(protein)
)

이 때문에 canonical BioMini에서는:

```text
DNA     → empty sequence 불허
RNA     → empty sequence 불허
Protein → empty sequence 허용
```

정책을 사용합니다.

여기서 중요한 교훈은 다음입니다.

> validation rule은 단순한 programming preference가 아니라
> scientific semantics와 계산 결과의 의미에 따라 결정되어야 한다.

# 12. Translation Frame

Phase I translation은 항상 **frame 0**을 사용합니다.

예를 들어:

```text
AUGGCCUUU
```

를 다음처럼 읽습니다.

```text
AUG | GCC | UUU
```

현재 Phase I에서는:

```text
frame 1
UGG | CCU | ...

frame 2
GGC | CUU | ...
```

를 자동 탐색하지 않습니다.

또한 ORF discovery도 수행하지 않습니다.

이것은 biology를 완전히 모델링한다는 의미가 아니라,
Phase I의 scope를 명확하게 제한한 것입니다.

# 13. Coding Strand Assumption

BioMini Phase I의 `DNA.transcribe()`는 저장된 DNA sequence를
**coding strand**로 간주합니다.

따라서:

```text
DNA: ATG
RNA: AUG
```

입니다.

template strand를 입력했다고 자동 판단하지 않습니다.

이 역시 명시적인 API semantic입니다.

# 14. Self-Check

In [ ]:
dna = DNA(
    "test",
    "ATGGCCTTT"
)

assert dna.complement() == "TACCGGAAA"
assert dna.reverse_complement() == "AAAGGCCAT"

rna = dna.transcribe()

assert isinstance(rna, RNA)
assert rna.get_sequence() == "AUGGCCUUU"

protein = rna.translate()

assert isinstance(protein, Protein)
assert protein.get_sequence() == "MAF"

stop_rna = RNA(
    "stop",
    "AUGUAA"
)

assert (
    stop_rna.translate().get_sequence()
    == "M"
)

incomplete_rna = RNA(
    "incomplete",
    "AUGGC"
)

assert (
    incomplete_rna.translate().get_sequence()
    == "M"
)

empty_product = RNA(
    "empty_product",
    "UAA"
).translate()

assert (
    empty_product.get_sequence()
    == ""
)

print(
    "Notebook 03 self-check: PASS"
)

# 15. Canonical BioMini와 비교

실제 BioMini package가 설치되어 있다면 같은 flow를 실행할 수 있습니다.

In [ ]:
try:
    from biomini import (
        DNA as BioMiniDNA,
        RNA as BioMiniRNA,
        Protein as BioMiniProtein,
    )

    bm_dna = BioMiniDNA(
        "gene1",
        "ATGGCCTTT"
    )

    bm_rna = bm_dna.transcribe()
    bm_protein = bm_rna.translate()

    print(
        "Canonical DNA     :",
        bm_dna
    )

    print(
        "Canonical RNA     :",
        bm_rna
    )

    print(
        "Canonical Protein :",
        bm_protein
    )

    print(
        "Protein sequence  :",
        bm_protein.get_sequence()
    )

except ImportError:
    print(
        "BioMini package is not installed in this environment.\n"
        "From the repository root, run:\n"
        "    python -m pip install -e ."
    )

# 16. Educational Version과 Canonical Version의 차이

이번 Notebook의 educational version은 transformation logic의 핵심만 보여줍니다.

canonical BioMini에서는 추가로 **provenance와 lineage**를 기록합니다.

예:

```text
DNA
id = D1
 ↓ transcription

RNA
parent_ids = [D1]
 ↓ translation

Protein
parent_ids = [RNA_ID]
```

즉 production BioMini에서는:

```text
무엇이 생성되었는가?
```

뿐 아니라:

```text
무엇으로부터 생성되었는가?
```

도 추적할 수 있습니다.

이 내용은 Notebook 07에서 본격적으로 다룹니다.

# 17. What Belongs in a Domain Object?

이번 Notebook에서 다음 behavior는 domain object 안에 두었습니다.

```text
DNA
- complement
- reverse_complement
- transcribe

RNA
- translate
```

왜냐하면 이들은 object의 biological identity와 직접 연결된 transformation이기 때문입니다.

하지만 다음 기능은 어떨까요?

```text
GC content
molecular weight
hydrophobic fraction
amino-acid composition
```

이들을 모두 DNA / RNA / Protein 안에 계속 추가하면
domain class가 점점 커집니다.

이 문제가 다음 Notebook의 핵심입니다.

# 18. Bridge to Notebook 04

다음 질문을 생각해 봅니다.

```python
protein.molecular_weight()
protein.hydrophobic_fraction()
protein.amino_acid_fraction()
protein.composition()
...
```

계속 이렇게 method를 추가해도 될까요?

처음에는 편하지만 곧 다음 문제가 생깁니다.

```text
Domain Identity
+
Biological Transformation
+
Analysis
+
Statistics
+
Feature Engineering
```

이 모든 책임이 한 object에 모입니다.

따라서 Notebook 04에서는:

```text
Protein
        ↓ has-a
ProteinAnalyzer
```

형태로 **domain object와 analysis logic을 분리**합니다.

# 19. What We Learned

이번 Notebook의 핵심 흐름:

```text
Sequence Object
    ↓
DNA complement
    ↓
reverse complement
    ↓
DNA → RNA
    ↓
RNA → Protein
    ↓
stop-codon policy
    ↓
incomplete-codon policy
    ↓
empty translation product
    ↓
explicit biological semantics
```

핵심 설계 원칙은 다음입니다.

> biological transformation을 단순한 string manipulation으로만 보지 않고,
> domain object 사이의 의미 있는 transformation으로 모델링한다.

그리고 scientific software에서는 edge case를 숨기지 않고
**명시적인 policy로 정의**해야 합니다.

# 20. Exercises

## Exercise 1 — Complement
다음 DNA sequence의 complement와 reverse complement를 계산하십시오.

```text
ATGCGTAC
```

직접 손으로 계산한 결과와 method 결과를 비교하십시오.

---

## Exercise 2 — Transcription
다음 DNA를 transcribe하십시오.

```text
ATGGGCTAA
```

생성된 RNA object의 type과 sequence를 확인하십시오.

---

## Exercise 3 — Translation
다음 RNA를 translation하십시오.

```text
AUGGCCUUU
```

codon별로 어떤 amino acid가 생성되는지 적으십시오.

---

## Exercise 4 — Stop Codon
다음을 비교하십시오.

```text
AUGGCCUAA
AUGGCCUAG
AUGGCCUGA
```

세 경우의 Protein sequence가 같은지 확인하십시오.

---

## Exercise 5 — Incomplete Codon
다음을 각각 translation하십시오.

```text
AUG
AUGG
AUGGC
AUGGCC
```

왜 그런 결과가 나오는지 설명하십시오.

---

## Exercise 6 — Empty Product
다음을 실행하십시오.

```text
UAA
UAG
UGA
```

각각 생성되는 Protein length를 확인하십시오.

---

## Exercise 7 — Design Exercise
현재 `complement()`는 `str`을 반환합니다.

대신 새로운 `DNA` object를 반환하도록 설계할 수도 있습니다.

두 방식의 장단점을 비교해 보십시오.

---

## Exercise 8 — Scope Question
BioMini Phase I에서 ORF search를 구현하지 않는 이유를
software scope와 educational scope 관점에서 설명해 보십시오.